In [1]:
import pandas as pd
import random

random.seed(42)

complaints = [
    # Roads
    ("Large pothole on main road causing accidents", "Roads", "High"),
    ("Road completely broken near school zone", "Roads", "High"),
    ("Street has cracks and bumps everywhere", "Roads", "Medium"),
    ("Minor road damage near park", "Roads", "Low"),
    ("Footpath broken and dangerous for walkers", "Roads", "Medium"),
    ("Pothole filled with water near hospital", "Roads", "High"),
    ("Road marking faded near signal", "Roads", "Low"),
    ("Speed breaker damaged and sharp", "Roads", "Medium"),

    # Water
    ("No water supply for 3 days in our area", "Water", "High"),
    ("Water pipe burst flooding the street", "Water", "High"),
    ("Dirty water coming from tap", "Water", "High"),
    ("Low water pressure in morning", "Water", "Medium"),
    ("Water leakage from underground pipe", "Water", "Medium"),
    ("Water supply irregular for a week", "Water", "High"),
    ("Water meter not working properly", "Water", "Low"),
    ("Sewage mixing with drinking water", "Water", "High"),

    # Electricity
    ("No electricity for 2 days in entire street", "Electricity", "High"),
    ("Street light not working for a month", "Electricity", "Medium"),
    ("Electric pole fallen on road", "Electricity", "High"),
    ("Sparking wire hanging dangerously", "Electricity", "High"),
    ("Power cut every day for 4 hours", "Electricity", "High"),
    ("Electricity bill incorrect this month", "Electricity", "Low"),
    ("Transformer making loud noise", "Electricity", "Medium"),
    ("Street lights on during daytime", "Electricity", "Low"),

    # Garbage
    ("Garbage not collected for 10 days", "Garbage", "High"),
    ("Garbage bin overflowing near market", "Garbage", "High"),
    ("Dead animals lying on road not removed", "Garbage", "High"),
    ("Garbage burning causing smoke", "Garbage", "Medium"),
    ("No dustbin in our locality", "Garbage", "Medium"),
    ("Garbage truck not coming regularly", "Garbage", "Medium"),
    ("Waste dumped near school", "Garbage", "High"),
    ("Littering on public road", "Garbage", "Low"),

    # Drainage
    ("Drain blocked causing flooding in homes", "Drainage", "High"),
    ("Sewage overflow on main road", "Drainage", "High"),
    ("Bad smell from open drain", "Drainage", "Medium"),
    ("Drain cover missing dangerous for children", "Drainage", "High"),
    ("Stagnant water breeding mosquitoes", "Drainage", "High"),
    ("Drain cleaning not done for months", "Drainage", "Medium"),
    ("Minor drain blockage near house", "Drainage", "Low"),
    ("Rainwater not draining properly", "Drainage", "Medium"),
]

# Multiply to create bigger dataset
data = complaints * 25
random.shuffle(data)

df = pd.DataFrame(data, columns=['complaint_text', 'category', 'priority'])
df.to_csv('../data/grievances.csv', index=False)

print(f"Dataset created: {df.shape}")
print(df['category'].value_counts())
print(df['priority'].value_counts())
df.head(10)

Dataset created: (1000, 3)
category
Electricity    200
Garbage        200
Water          200
Roads          200
Drainage       200
Name: count, dtype: int64
priority
High      500
Medium    325
Low       175
Name: count, dtype: int64


,complaint_text,category,priority
0,No electricity for 2 days in entire street,Electricity,High
1,Garbage burning causing smoke,Garbage,Medium
2,Sewage mixing with drinking water,Water,High
3,Street has cracks and bumps everywhere,Roads,Medium
4,Sewage overflow on main road,Drainage,High
5,Minor road damage near park,Roads,Low
6,Pothole filled with water near hospital,Roads,High
7,Waste dumped near school,Garbage,High
8,Bad smell from open drain,Drainage,Medium
9,Minor road damage near park,Roads,Low


In [2]:
import pandas as pd
import re
import nltk

nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords

df = pd.read_csv('../data/grievances.csv')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['cleaned_text'] = df['complaint_text'].apply(clean_text)

print("✅ Text cleaned!")
print(df[['complaint_text', 'cleaned_text']].head(5))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


✅ Text cleaned!
                               complaint_text                    cleaned_text
0  No electricity for 2 days in entire street  electricity days entire street
1               Garbage burning causing smoke   garbage burning causing smoke
2           Sewage mixing with drinking water    sewage mixing drinking water
3      Street has cracks and bumps everywhere  street cracks bumps everywhere
4                Sewage overflow on main road       sewage overflow main road


In [3]:
from sklearn.preprocessing import LabelEncoder

le_category = LabelEncoder()
le_priority  = LabelEncoder()

df['category_encoded'] = le_category.fit_transform(df['category'])
df['priority_encoded']  = le_priority.fit_transform(df['priority'])

print("Categories:", list(le_category.classes_))
print("Priorities:", list(le_priority.classes_))


Categories: ['Drainage', 'Electricity', 'Garbage', 'Roads', 'Water']
Priorities: ['High', 'Low', 'Medium']


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Vectorize text
tfidf = TfidfVectorizer(max_features=500)
X = tfidf.fit_transform(df['cleaned_text'])

# Category model
y_cat = df['category_encoded']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

# Priority model
y_pri = df['priority_encoded']
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y_pri, test_size=0.2, random_state=42
)

print("✅ TF-IDF done!")
print("Feature matrix shape:", X.shape)

✅ TF-IDF done!
Feature matrix shape: (1000, 116)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Category classifier
cat_model = RandomForestClassifier(n_estimators=100, random_state=42)
cat_model.fit(X_train_c, y_train_c)
cat_pred = cat_model.predict(X_test_c)
print(f"✅ Category Accuracy: {accuracy_score(y_test_c, cat_pred)*100:.2f}%")
print(classification_report(y_test_c, cat_pred, target_names=le_category.classes_))

# Priority classifier
pri_model = RandomForestClassifier(n_estimators=100, random_state=42)
pri_model.fit(X_train_p, y_train_p)
pri_pred = pri_model.predict(X_test_p)
print(f"✅ Priority Accuracy: {accuracy_score(y_test_p, pri_pred)*100:.2f}%")
print(classification_report(y_test_p, pri_pred, target_names=le_priority.classes_))

✅ Category Accuracy: 100.00%
              precision    recall  f1-score   support

    Drainage       1.00      1.00      1.00        48
 Electricity       1.00      1.00      1.00        39
     Garbage       1.00      1.00      1.00        42
       Roads       1.00      1.00      1.00        34
       Water       1.00      1.00      1.00        37

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

✅ Priority Accuracy: 100.00%
              precision    recall  f1-score   support

        High       1.00      1.00      1.00       100
         Low       1.00      1.00      1.00        37
      Medium       1.00      1.00      1.00        63

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [6]:
import joblib

joblib.dump(cat_model,   '../model/category_model.pkl')
joblib.dump(pri_model,   '../model/priority_model.pkl')
joblib.dump(tfidf,       '../model/tfidf_vectorizer.pkl')
joblib.dump(le_category, '../model/label_category.pkl')
joblib.dump(le_priority, '../model/label_priority.pkl')

print("✅ All models saved!")

✅ All models saved!
